In [1]:
import os
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, dayofweek, to_timestamp, count, when
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# ==========================================
# 1. CONFIGURATION PATHS (Sesuaikan di Sini)
# ==========================================
# Gunakan relative path atau absolute path lokal Anda
BASE_DIR = "." 
CSV_PATH = os.path.join(BASE_DIR, "HI-Medium_Trans.csv")
PARQUET_PATH = os.path.join(BASE_DIR, "HI-Medium_Trans.parquet")
TRAIN_PARQUET_PATH = os.path.join(BASE_DIR, "train_data_sampled.parquet")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "catboost_aml_model.cbm")

if os.path.exists(CSV_PATH):
    print(f"[+] Data ditemukan, siap digunakan di: {CSV_PATH}")
else:
    raise FileNotFoundError(f"[-] File CSV tidak ditemukan di: {CSV_PATH}. Silakan pindahkan file atau ubah path-nya.")

[+] Data ditemukan, siap digunakan di: .\HI-Medium_Trans.csv


In [2]:
# ==========================================
# 2. INITIALIZE SPARK SESSION
# ==========================================
if 'spark' in locals():
    spark.stop()

print("[*] Menginisialisasi Spark Session dengan Optimasi Memori Lokal...")
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IBM-AML-Local-Pipeline") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.shuffle.partitions", "400") \
    .getOrCreate()

print("[+] Spark Session Aktif")

[*] Menginisialisasi Spark Session dengan Optimasi Memori Lokal...
[+] Spark Session Aktif


In [3]:
# # ==========================================
# # 3. CONVERT CSV TO PARQUET
# # ==========================================
# print("[*] Memproses konversi seluruh kolom ke Parquet...")
# start_time = time.time()

# df_initial = spark.read.csv(CSV_PATH, header=True, inferSchema=True)
# df_initial.write.mode("overwrite").parquet(PARQUET_PATH)

# print(f"[+] KONVERSI SELESAI dalam {time.time() - start_time:.2f} detik!")

In [4]:
# ==========================================
# 4. EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
print("[*] Memuat data dari format Parquet lokal...")
df = spark.read.parquet(PARQUET_PATH)

print(f"[+] Total baris data: {df.count():,}")
print("\n[+] Struktur Kolom dan Tipe Data:")
df.printSchema()

print("\n[*] Menghitung distribusi kelas target (Is Laundering)...")
df.groupBy("Is Laundering").count().show()

print("\n[*] Mengecek jumlah nilai Null/Missing Values di setiap kolom...")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

[*] Memuat data dari format Parquet lokal...
[+] Total baris data: 31,898,238

[+] Struktur Kolom dan Tipe Data:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)


[*] Menghitung distribusi kelas target (Is Laundering)...
+-------------+--------+
|Is Laundering|   count|
+-------------+--------+
|            0|31863008|
|            1|   35230|
+-------------+--------+


[*] Mengecek jumlah nilai Null/Missing Values di setiap kolom...
+---------+---------+--------+-------+--------+---------------+------------------+-----------+----------------+-----------

In [5]:
# ==========================================
# 5. FEATURE ENGINEERING (CLEAN VERSION)
# ==========================================
from pyspark.sql.functions import col, hour, dayofweek, to_timestamp, hash, abs
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

print("[*] Memulai tahapan Feature Engineering...")

# 1. Ekstrak Waktu dari Kolom Timestamp (Menggunakan kurung () agar aman tanpa backslash)
df_cleaned = (df.withColumn("Parsed_Time", to_timestamp(col("Timestamp"), "yyyy/MM/dd HH:mm"))
               .withColumn("Hour", hour(col("Parsed_Time")))
               .withColumn("DayOfWeek", dayofweek(col("Parsed_Time")))
               .drop("Timestamp", "Parsed_Time"))

# 2. StringIndexer untuk kolom kardinalitas rendah (tanpa backslash pada "keep")
kolom_kategori_rendah = ["Payment Format", "Payment Currency", "Receiving Currency"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_Indexed", handleInvalid="keep") for c in kolom_kategori_rendah]

pipeline = Pipeline(stages=indexers)
print("[*] Melakukan indexing pada kolom teks kardinalitas rendah...")
df_transformed = pipeline.fit(df_cleaned).transform(df_cleaned)

# 3. BYPASS METADATA: Gunakan fungsi Hash untuk kolom akun dengan kardinalitas tinggi
print("[*] Melakukan transformasi hashing cepat untuk Account2 dan Account4...")
df_final = (df_transformed
    .withColumn("Account2_Indexed", abs(hash(col("Account2"))).cast("double"))
    .withColumn("Account4_Indexed", abs(hash(col("Account4"))).cast("double"))
)

# 4. Hapus kolom teks asli yang sudah tidak digunakan
kolom_asli_drop = ["Payment Format", "Payment Currency", "Receiving Currency", "Account2", "Account4"]
df_final = df_final.drop(*kolom_asli_drop)

print("\n[+] FEATURE ENGINEERING SELESAI!")
print("[+] Struktur Dataframe Akhir siap (Metadata Bersih):")
df_final.printSchema()

[*] Memulai tahapan Feature Engineering...
[*] Melakukan indexing pada kolom teks kardinalitas rendah...
[*] Melakukan transformasi hashing cepat untuk Account2 dan Account4...

[+] FEATURE ENGINEERING SELESAI!
[+] Struktur Dataframe Akhir siap (Metadata Bersih):
root
 |-- From Bank: integer (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Is Laundering: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Payment Format_Indexed: double (nullable = false)
 |-- Payment Currency_Indexed: double (nullable = false)
 |-- Receiving Currency_Indexed: double (nullable = false)
 |-- Account2_Indexed: double (nullable = false)
 |-- Account4_Indexed: double (nullable = false)



In [6]:
# ==========================================
# 6. SAVE PREPROCESSED DATA TO DISK (Lineage Break)
# ==========================================
PARQUET_PREPROCESSED_PATH = "HI-Medium_Trans_Final.parquet"

print("[*] Menyimpan df_final ke disk untuk memutus Lineage (Menghemat RAM Lokal)...")
df_final.write.mode("overwrite").parquet(PARQUET_PREPROCESSED_PATH)

print("[+] Data feature engineering berhasil dikunci aman di disk!")

[*] Menyimpan df_final ke disk untuk memutus Lineage (Menghemat RAM Lokal)...
[+] Data feature engineering berhasil dikunci aman di disk!


In [7]:
# ==========================================
# 7. DOWNSAMPLING & PREPARATION DATA
# ==========================================
print("[*] Memuat ulang data preprocessed yang bersih dari disk...")
df_final_clean = spark.read.parquet("HI-Medium_Trans_Final.parquet")

print("Menggabungkan kolom fitur menjadi satu vektor tunggal...")
kolom_fitur = [
    'From Bank', 'To Bank', 'Amount Received', 'Amount Paid',
    'Hour', 'DayOfWeek', 'Payment Format_Indexed',
    'Payment Currency_Indexed', 'Receiving Currency_Indexed',
    'Account2_Indexed', 'Account4_Indexed'
]

assembler = VectorAssembler(inputCols=kolom_fitur, outputCol="features")
df_ml = assembler.transform(df_final_clean) # Menggunakan df_final_clean

# Pilih fitur dan label
df_ml_final = df_ml.select("features", col("Is Laundering").alias("label"))

print("Membagi data menjadi Train Set (80%) dan Test Set (20%)...")
train_data, test_data = df_ml_final.randomSplit([0.8, 0.2], seed=42)

print("\n[+] PERSIAPAN DATA SELESAI!")

[*] Memuat ulang data preprocessed yang bersih dari disk...
Menggabungkan kolom fitur menjadi satu vektor tunggal...
Membagi data menjadi Train Set (80%) dan Test Set (20%)...

[+] PERSIAPAN DATA SELESAI!


In [8]:
# ==========================================
# 8. TRAINING MODEL CATBOOST (FIXED ARROW OOM)
# ==========================================
from pyspark.ml.functions import vector_to_array

print("[*] Melakukan downsampling data khusus untuk model...")

# Pisahkan kelas target
train_laundering = train_data.filter(train_data.label == 1)
train_normal = train_data.filter(train_data.label == 0)

# Hitung fraksi sampel secara aman
count_normal = train_normal.count()
fraction = 100000 / count_normal
train_normal_sampled = train_normal.sample(withReplacement=False, fraction=fraction, seed=42)

# Gabung kedua kelas di Spark DULU (lebih efisien)
train_combined = train_laundering.union(train_normal_sampled)

# KONVERSI KOLOM VEKTOR KE ARRAY NUMERIK SEBELUM toPandas()
# Ini menghindari OOM karena VectorUDT tidak di-serialize ke Python
train_combined = train_combined.withColumn("features_array", vector_to_array("features")) \
                               .drop("features")

print("\n[*] Mengonversi data ke Pandas...")
start_conv = time.time()

# Matikan Arrow optimization untuk keamanan
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

train_pdf = train_combined.toPandas()

# Nyalakan kembali Arrow
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print(f"[+] Konversi selesai dalam {time.time() - start_conv:.2f} detik!")
print(f"[+] Total data training siap proses CatBoost: {len(train_pdf):,} baris")

# Ekstrak fitur & label menjadi array NumPy
X_train = np.array(train_pdf['features_array'].tolist())
y_train = train_pdf['label'].values
print("[+] X_train dan y_train berhasil disiapkan!")


[*] Melakukan downsampling data khusus untuk model...

[*] Mengonversi data ke Pandas...
[+] Konversi selesai dalam 61.95 detik!
[+] Total data training siap proses CatBoost: 128,306 baris
[+] X_train dan y_train berhasil disiapkan!


In [9]:
# ==========================================
# 8b. TRAINING MODEL CATBOOST & SAVE
# ==========================================
print("[*] Melatih model CatBoost Classifier...")
start_train = time.time()

catboost_model = CatBoostClassifier(
    iterations=700,
    learning_rate=0.05,
    depth=6,
    task_type="CPU",  # Ubah ke "GPU" jika lokal Anda sudah terpasang CUDA + NVIDIA GPU
    random_seed=42,
    verbose=100
)
catboost_model.fit(X_train, y_train)
print(f"[+] Training selesai dalam {time.time() - start_train:.2f} detik!")

# Menyimpan Model secara lokal
catboost_model.save_model(MODEL_SAVE_PATH)
print(f"[+] Model berhasil disimpan lokal di: {MODEL_SAVE_PATH}")


[*] Melatih model CatBoost Classifier...
0:	learn: 0.6012680	total: 170ms	remaining: 1m 58s
100:	learn: 0.1857059	total: 1.98s	remaining: 11.8s
200:	learn: 0.1763494	total: 3.81s	remaining: 9.47s
300:	learn: 0.1696562	total: 5.6s	remaining: 7.42s
400:	learn: 0.1662424	total: 7.35s	remaining: 5.48s
500:	learn: 0.1635477	total: 9.14s	remaining: 3.63s
600:	learn: 0.1612433	total: 10.9s	remaining: 1.8s
699:	learn: 0.1592831	total: 12.8s	remaining: 0us
[+] Training selesai dalam 13.17 detik!
[+] Model berhasil disimpan lokal di: .\catboost_aml_model.cbm


In [10]:
# ==========================================
# 9. EVALUASI DATA TEST
# ==========================================
from pyspark.ml.functions import vector_to_array

print("\n[*] Mulai persiapan data evaluasi...")
test_laundering = test_data.filter(test_data.label == 1)
test_normal = test_data.filter(test_data.label == 0)

test_normal_sampled = test_normal.sample(withReplacement=False, fraction=0.024, seed=42)
test_data_safe = test_laundering.union(test_normal_sampled)

print(f"[+] Data testing disederhanakan menjadi: {test_data_safe.count():,} baris.")

# Konversi VectorUDT ke Array sebelum toPandas
test_data_safe = test_data_safe.withColumn("features_array", vector_to_array("features")) \
                               .drop("features")

print("[*] Melakukan Prediksi menggunakan CatBoost...")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
test_pdf = test_data_safe.toPandas()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

X_test = np.array(test_pdf['features_array'].tolist())

preds = catboost_model.predict(X_test)
probs = catboost_model.predict_proba(X_test)[:, 1]

res_pdf = pd.DataFrame({
    'label': test_pdf['label'].values,
    'prediction': preds.astype(float),
    'rawPrediction': probs
})
predictions_safe = spark.createDataFrame(res_pdf)

print("[*] Menghitung skor akhir ROC-AUC...")
evaluator_safe = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_safe = evaluator_safe.evaluate(predictions_safe)
print(f"\n[+] CatBoost ROC-AUC: {auc_safe:.4f}")



[*] Mulai persiapan data evaluasi...
[+] Data testing disederhanakan menjadi: 160,347 baris.
[*] Melakukan Prediksi menggunakan CatBoost...
[*] Menghitung skor akhir ROC-AUC...

[+] CatBoost ROC-AUC: 0.9742


In [11]:
# ==========================================
# 10. CONFUSION MATRIX & METRIKS
# ==========================================
print("\n[*] Menghitung Confusion Matrix...")
confusion_matrix = predictions_safe.groupBy("label").pivot("prediction", [0.0, 1.0]).count().na.fill(0)
confusion_matrix.show()

data_cm = confusion_matrix.collect()
tn, fp, fn, tp = 0, 0, 0, 0

for row in data_cm:
    if row['label'] == 0:
        tn = row['0.0']
        fp = row['1.0']
    elif row['label'] == 1:
        fn = row['0.0']
        tp = row['1.0']

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tn + fp + fn + tp)

print("="*45)
print(f" Classification Report - CatBoost (Lokal)")
print("="*45)
print(f"Accuracy     : {accuracy:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall (TPR) : {recall:.4f}")
print(f"F1-Score     : {f1_score:.4f}")
print("="*45)
print(f"Detail: True Positive={tp}, False Positive={fp}, False Negative={fn}, True Negative={tn}")


[*] Menghitung Confusion Matrix...
+-----+------+----+
|label|   0.0| 1.0|
+-----+------+----+
|    0|146746|6356|
|    1|  1343|5902|
+-----+------+----+

 Classification Report - CatBoost (Lokal)
Accuracy     : 0.9520
Precision    : 0.4815
Recall (TPR) : 0.8146
F1-Score     : 0.6052
Detail: True Positive=5902, False Positive=6356, False Negative=1343, True Negative=146746
